In [1]:
import os

print(os.listdir('/kaggle/input'))

['datasets']


In [2]:
print(os.listdir('/kaggle/input/datasets'))

['khanfashee']


In [3]:
print(os.listdir('/kaggle/input/datasets/khanfashee'))

['nih-chest-x-ray-14-224x224-resized']


In [4]:
base_path = '/kaggle/input/datasets/khanfashee/nih-chest-x-ray-14-224x224-resized'

print(os.listdir(base_path))

['images-224', 'test_list_NIH.txt', 'BBox_List_2017_Official_NIH.csv', 'train_val_list_NIH.txt', 'Data_Entry_2017.csv', 'pretrained_model.h5']


In [5]:
import pandas as pd

csv_path = os.path.join(base_path, 'Data_Entry_2017.csv')

df = pd.read_csv(csv_path)

print(df.shape)
print(df.columns.tolist())

df.head()

(112120, 12)
['Image Index', 'Finding Labels', 'Follow-up #', 'Patient ID', 'Patient Age', 'Patient Gender', 'View Position', 'OriginalImage[Width', 'Height]', 'OriginalImagePixelSpacing[x', 'y]', 'Unnamed: 11']


,Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y],Unnamed: 11
0,00000001_000.png,Cardiomegaly,0,1,058Y,M,PA,2682,2749,0.143,0.143,NaN
1,00000001_001.png,Cardiomegaly|Emphysema,1,1,058Y,M,PA,2894,2729,0.143,0.143,NaN
2,00000001_002.png,Cardiomegaly|Effusion,2,1,058Y,M,PA,2500,2048,0.168,0.168,NaN
3,00000002_000.png,No Finding,0,2,081Y,M,PA,2500,2048,0.171,0.171,NaN
4,00000003_000.png,Hernia,0,3,081Y,F,PA,2582,2991,0.143,0.143,NaN


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112120 entries, 0 to 112119
Data columns (total 12 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   Image Index                  112120 non-null  object 
 1   Finding Labels               112120 non-null  object 
 2   Follow-up #                  112120 non-null  int64  
 3   Patient ID                   112120 non-null  int64  
 4   Patient Age                  112120 non-null  object 
 5   Patient Gender               112120 non-null  object 
 6   View Position                112120 non-null  object 
 7   OriginalImage[Width          112120 non-null  int64  
 8   Height]                      112120 non-null  int64  
 9   OriginalImagePixelSpacing[x  112120 non-null  float64
 10  y]                           112120 non-null  float64
 11  Unnamed: 11                  0 non-null       float64
dtypes: float64(3), int64(4), object(5)
memory usage: 10.3+ MB


In [7]:
df.isnull().sum()

Image Index                         0
Finding Labels                      0
Follow-up #                         0
Patient ID                          0
Patient Age                         0
Patient Gender                      0
View Position                       0
OriginalImage[Width                 0
Height]                             0
OriginalImagePixelSpacing[x         0
y]                                  0
Unnamed: 11                    112120
dtype: int64

In [8]:
df = df.drop(columns=["Unnamed: 11"])

print(df.shape)
print(df.columns.tolist())

(112120, 11)
['Image Index', 'Finding Labels', 'Follow-up #', 'Patient ID', 'Patient Age', 'Patient Gender', 'View Position', 'OriginalImage[Width', 'Height]', 'OriginalImagePixelSpacing[x', 'y]']


In [9]:
num_images = len(df)
num_patients = df["Patient ID"].nunique()

print("Number of images:", num_images)
print("Number of patients:", num_patients)

Number of images: 112120
Number of patients: 30805


In [10]:
images_per_patient = df["Patient ID"].value_counts()

print(images_per_patient.describe())

count    30805.000000
mean         3.639669
std          7.266734
min          1.000000
25%          1.000000
50%          1.000000
75%          3.000000
max        184.000000
Name: count, dtype: float64


### Patient Statistics

- Total images: 112,120
- Unique patients: 30,805
- Mean images per patient: 3.64
- Median images per patient: 1
- Maximum images for a single patient: 184

Since multiple images can belong to the same patient, the dataset should be split by Patient ID to prevent data leakage.

In [11]:
print("Patient Gender")
print(df["Patient Gender"].value_counts())

print("\nView Position")
print(df["View Position"].value_counts())

Patient Gender
Patient Gender
M    63340
F    48780
Name: count, dtype: int64

View Position
View Position
PA    67310
AP    44810
Name: count, dtype: int64


In [12]:
gender_ratio = df["Patient Gender"].value_counts(normalize=True) * 100
view_ratio = df["View Position"].value_counts(normalize=True) * 100

print("Patient Gender (%)")
print(gender_ratio.round(2))

print("\nView Position (%)")
print(view_ratio.round(2))

Patient Gender (%)
Patient Gender
M    56.49
F    43.51
Name: proportion, dtype: float64

View Position (%)
View Position
PA    60.03
AP    39.97
Name: proportion, dtype: float64


### Gender and View Position

- Male: 56.49%
- Female: 43.51%
- PA view: 60.03%
- AP view: 39.97%

These distributions are based on X-ray images, not unique patients.

In [13]:
print(df["Patient Age"].head(10))

print("\nAge suffixes:")
print(df["Patient Age"].astype(str).str[-1].value_counts())

0    058Y
1    058Y
2    058Y
3    081Y
4    081Y
5    074Y
6    075Y
7    076Y
8    077Y
9    078Y
Name: Patient Age, dtype: object

Age suffixes:
Patient Age
Y    112093
M        26
D         1
Name: count, dtype: int64


In [14]:
age_numeric = pd.to_numeric(
    df["Patient Age"].astype(str).str.extract(r"(\d+)")[0],
    errors="coerce"
)

print(age_numeric.describe())

count    112120.000000
mean         46.901463
std          16.839923
min           1.000000
25%          35.000000
50%          49.000000
75%          59.000000
max         414.000000
Name: 0, dtype: float64


In [15]:
age_number = pd.to_numeric(
    df["Patient Age"].astype(str).str.extract(r"(\d+)")[0],
    errors="coerce"
)

df.loc[age_number.nlargest(10).index, ["Patient Age", "Patient ID", "Image Index"]]

,Patient Age,Patient ID,Image Index
46965,414Y,11973,00011973_002.png
86264,413Y,21275,00021275_003.png
20852,412Y,5567,00005567_000.png
85404,412Y,21047,00021047_002.png
91369,412Y,22811,00022811_000.png
84810,411Y,20900,00020900_002.png
101194,155Y,26871,00026871_000.png
104590,155Y,27989,00027989_000.png
98495,154Y,26028,00026028_001.png
95794,153Y,25206,00025206_000.png


In [16]:
df[
    df["Patient Age"].astype(str).str[-1].isin(["M", "D"])
][["Patient Age", "Patient ID", "Image Index"]]

,Patient Age,Patient ID,Image Index
53031,011M,13378,00013378_000.png
69369,018M,17121,00017121_001.png
77180,018M,18962,00018962_000.png
77502,011M,19045,00019045_002.png
77503,013M,19045,00019045_003.png
77504,020M,19045,00019045_004.png
77505,021M,19045,00019045_005.png
79441,007M,19515,00019515_000.png
79442,018M,19515,00019515_001.png
79443,019M,19515,00019515_002.png


In [17]:
age_unit = df["Patient Age"].astype(str).str[-1]

age_value = pd.to_numeric(
    df["Patient Age"].astype(str).str.extract(r"(\d+)")[0],
    errors="coerce"
)

invalid_year_age = df[
    (age_unit == "Y") & (age_value > 100)
][["Patient Age", "Patient ID", "Image Index"]]

print("Year ages over 100:", len(invalid_year_age))
invalid_year_age

Year ages over 100: 16


,Patient Age,Patient ID,Image Index
20852,412Y,5567,00005567_000.png
46965,414Y,11973,00011973_002.png
48284,148Y,12238,00012238_010.png
55742,148Y,13950,00013950_000.png
58650,150Y,14520,00014520_026.png
62929,149Y,15558,00015558_000.png
74884,152Y,18366,00018366_044.png
78795,151Y,19346,00019346_000.png
84810,411Y,20900,00020900_002.png
85404,412Y,21047,00021047_002.png


### Age Data Quality

`Patient Age` contains values expressed in years (`Y`), months (`M`), and days (`D`).

Most records use years, but some implausible year values such as `414Y`, `413Y`, and `155Y` were found.  
These values should be treated as potential metadata errors and must not be used directly as patient age without preprocessing.

In [20]:
import numpy as np
def convert_age_to_years(age):
    age = str(age)

    value = int(age[:-1])
    unit = age[-1]

    if unit == "Y":
        # Treat implausible year values as missing
        if value > 100:
            return np.nan
        return float(value)

    elif unit == "M":
        return value / 12

    elif unit == "D":
        return value / 365.25

    return np.nan


df["Age Years"] = df["Patient Age"].apply(convert_age_to_years)

print(df["Age Years"].describe())
print("\nMissing/invalid ages:", df["Age Years"].isna().sum())

count    112104.000000
mean         46.869090
std          16.605933
min           0.002738
25%          35.000000
50%          49.000000
75%          59.000000
max          95.000000
Name: Age Years, dtype: float64

Missing/invalid ages: 16


### Age Preprocessing

Patient age values were converted to years.

- `Y` values were interpreted as years.
- `M` values were converted from months to years.
- `D` values were converted from days to years.
- Implausible ages greater than 100 years were treated as missing values.

After preprocessing:
- Valid age records: 112,104
- Invalid/missing age records: 16
- Mean age: 46.87 years
- Median age: 49 years
- Maximum valid age: 95 years